<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/rivaasrasr-finetuning-conformer-ctc-nemo/nvidia_logo.png" style="width: 90px; float: right;">

# How to Fine-Tune a Riva ASR Acoustic Model with NVIDIA NeMo
This tutorial walks you through how to fine-tune an NVIDIA Riva nemotron-asr-streaming-rnnt-prompt acoustic model with NVIDIA NeMo.

## NVIDIA Riva NIM Overview

NVIDIA Riva ASR NIM APIs provide easy access to state-of-the-art automatic speech recognition (ASR) models for multiple languages. Riva ASR NIM models are built on the NVIDIA software platform, incorporating CUDA, TensorRT, and Triton to offer out-of-the-box GPU acceleration.

In this tutorial, we will interact with the automated speech recognition (ASR) APIs.

For more information about Riva ASR NIM, refer to the [Riva NIM documentation](https://docs.nvidia.com/nim/riva/asr/latest/overview.html).

## NeMo (Neural Modules)
[NVIDIA NeMo](https://developer.nvidia.com/nvidia-nemo) is an open-source framework for building, training, and fine-tuning GPU-accelerated speech AI and NLU models with a simple Python interface. For information about how to set up NeMo, refer to the [NeMo GitHub](https://github.com/NVIDIA/NeMo) instructions.

In [1]:
"""
You can run either this tutorial locally (if you have all the dependencies and a GPU) or on Google Colab. If you run this tutorial in docker container with nemo supported you can skip this step.

Perform the following steps to setup in Google Colab:
1. Install cuda toolkit "conda install cudatoolkit"
2. Open a new Python 3 notebook. (3.8<=Python<=3.12)
3. Import this notebook from GitHub.
   a. Click **File** > **Upload Notebook** > **GITHUB** tab > copy/paste the GitHub URL.
4. Connect to an instance with a GPU.
   a. Click **Runtime** > Change the runtime type > select **GPU** for the hardware accelerator.
5. Run this cell to set up the dependencies.
6. Restart the runtime.
   a. Click **Runtime** > **Restart Runtime** for any upgraded packages to take effect.
"""

# Install Dependencies
!pip install wget
!apt-get install sox libsndfile1 ffmpeg libsox-fmt-mp3 jq
!pip install text-unidecode
!pip install matplotlib>=3.3.2
!pip install Cython
!pip3 install --no-cache-dir huggingface-hub==0.23.2

## Install NeMo
BRANCH = 'main'
!python -m pip install "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@{BRANCH}"

"""
Remember to restart the runtime for the kernel to pick up any upgraded packages (e.g. matplotlib)!
Alternatively, in the case where you want to use the "Run All Cells" (or similar) option,
uncomment `exit()` below to crash and restart the kernel.
"""
exit()

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=3133b25ff2caf3f20ff5ce3444312d46ba156f3f65ea10999b752d8f3c763bd9
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
jq is already the newest version (1.6-2.1ubuntu3.2).
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa
  libsox-fmt-base libsox3 libwavpack1
Suggested packages:
  libsox-fmt-all
The following NEW packages will be installed:
  libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa
  libsox-fmt-base libsox-fmt-mp3 libsox3 libwavpack1 sox
0 upgraded, 10 newly insta

---
## Fine-Tuning an ASR model with NeMo

### Download the Data

In this tutorial, we will use the popular AN4 dataset. Let's download it.

In [ ]:
! wget https://dldata-public.s3.us-east-2.amazonaws.com/an4_sphere.tar.gz  # for the original source, please visit http://www.speech.cs.cmu.edu/databases/an4/an4_sphere.tar.gz

After downloading, untar the dataset and move it to the correct directory.

In [1]:
import os

DATA_DIR = os.getcwd()
os.environ["DATA_DIR"] = DATA_DIR

In [2]:
from huggingface_hub import notebook_login

notebook_login()

### Pre-Processing

This step converts the `.mp3` files into `.wav` files and splits the data into training and testing sets. It also generates a "meta-data" file to be consumed by the data-loader for training and testing.

In [30]:
import json

import librosa
import soundfile as sf
from datasets import Audio, load_dataset

# For now, mos-BF is not supported, I will use fr-FR and figure out to add it later

def build_hf_manifest(dataset, manifest_path, target_wavs_dir,  audio_key='audio', text_key='text',
                      sample_rate=16000):
    """Build a NeMo manifest from a Hugging Face dataset."""
    os.makedirs(target_wavs_dir, exist_ok=True)

    with open(manifest_path, 'w') as fout:
        for i, item in enumerate(dataset):
            audio_data = item[audio_key]
            transcript = item[text_key].lower().strip()
            audio_path = os.path.join(target_wavs_dir, f"audio_{i}.wav")
            sf.write(audio_path, audio_data['array'], audio_data['sampling_rate'])

            duration = librosa.get_duration(path=audio_path, sr=sample_rate)

            metadata = {
                "audio_filepath": audio_path,
                "duration": duration,
                "text": transcript,
                "lang": "fr-FR", #TODO: Find mos-BF
                "target_lang": "fr-FR" #TODO: Find mos-BF
            }
            json.dump(metadata, fout)
            fout.write('\n')
    print(f"Manifest created at: {manifest_path}")

In [31]:
try:
    print("Loading Hugging Face dataset...")
    hf_dataset = load_dataset("madoss/faso-speech", "moore")

    hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=16000))

    hf_manifest_dir = os.path.join(DATA_DIR, 'hf_manifests')

    os.makedirs(hf_manifest_dir, exist_ok=True)
    hf_train_manifest_path = os.path.join(hf_manifest_dir, 'hf_train_manifest.json')
    hf_test_manifest_path = os.path.join(hf_manifest_dir, 'hf_test_manifest.json')
    build_hf_manifest(dataset=hf_dataset["train"], manifest_path=hf_train_manifest_path,
                      target_wavs_dir="data/train")
    build_hf_manifest(dataset=hf_dataset["validation"], manifest_path=hf_test_manifest_path,
                          target_wavs_dir="data/validation")

    print(f"First 5 lines of the generated manifest: {hf_train_manifest_path}")
    with open(hf_train_manifest_path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(line.strip())
except Exception as e:
    print(f"Error loading or processing Hugging Face dataset: {e}")

Loading Hugging Face dataset...
Error loading or processing Hugging Face dataset: getFramesPlayedInRangeAudio, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1181, No audio frames were decoded. This is probably because start_seconds is too high(0),or because stop_seconds(nullopt) is too low.


Let's listen to a sample audio file.

In [6]:
# change path of the file here
import os

import IPython.display as ipd

path = os.environ["DATA_DIR"] + '/data/train/audio_1.wav'
ipd.Audio(path)

### Tokenizers

If training dataset is not enough (<50 hrs) we recommend using tokenizers from pretrained model itself. The following tutorials explains how to use tokenizers from pretrained models for finetuning. If there's a change in vocab or you wish to train your own tokenizers you can use [NeMo tokenizer training script](https://github.com/NVIDIA/NeMo/blob/main/scripts/tokenizers/process_asr_text_tokenizer.py) and use [Hybrid model training script](https://github.com/NVIDIA/NeMo/blob/main/examples/asr/asr_hybrid_transducer_ctc/speech_to_text_hybrid_rnnt_ctc_bpe.py) to finetune the model on your data. Refer [asr-finetune-conformer-nemo](https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-conformer-ctc-nemo.ipynb) tutorial for more details.


#### Training nemotron-asr-streaming-rnnt-multilingual-prompt model

NeMo uses `.yaml` files to configure the training parameters. You may update them directly by editing the configuration file or from the command-line interface. For example, if the number of epochs needs to be modified, along with a change in the learning rate, you can add `trainer.max_epochs=100` and `optim.lr=0.02` and train the model.

The following sample command uses the `speech_to_text_finetune.py` script in the `examples` folder to train/fine-tune a nemotron-asr-streaming-rnnt-multilingual ASR model for 1 epoch. For other ASR models like Citrinet, Conformer, you may find the appropriate config files in the NeMo GitHub repo under [examples/asr/conf/](https://github.com/NVIDIA/NeMo/tree/main/examples/asr/conf).


In [21]:
from huggingface_hub import hf_hub_download

HF_CKPT = hf_hub_download(
    repo_id="nvidia/nemotron-3.5-asr-streaming-0.6b",
    filename="nemotron-3.5-asr-streaming-0.6b.nemo",
    local_dir="/opt/prompt_model/"
)

print(HF_CKPT)  # should print the full path to the .nemo file

nemotron-3.5-asr-streaming-0.6b.nemo:   0%|          | 0.00/2.37G [00:00<?, ?B/s]

/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b.nemo


In [ ]:
# To fully train the model, you'll need to increase trainer.max_epochs from 1.
# Empirical evidence suggests that around 200 epochs should suffice.
# If you are getting out of memory errors, reduce the ++model.train_ds.batch_duration value
NEMO_DIR = '/opt/prompt_model/NeMo'
BRANCH = 'main' # Subject to change to a fixed tag / version upon release
# ! git clone -b $BRANCH https://github.com/NVIDIA-NeMo/NeMo $NEMO_DIR

os.environ["PYTHONPATH"] = NEMO_DIR + ":" + os.environ.get("PYTHONPATH", "")
os.environ["NEMO_DIR"] = NEMO_DIR # we need because of shell use because I got python3: can't open file '/{NEMO_DIR}/examples/asr/speech_to_text_finetune.py': [Errno 2] No such file or directory

os.environ["HF_CKPT"] = HF_CKPT

! python {NEMO_DIR}/examples/asr/speech_to_text_finetune.py \
        --config-path="../asr/conf/fastconformer/cache_aware_streaming" --config-name=fastconformer_transducer_bpe_streaming_prompt.yaml \
        +init_from_nemo_model={HF_CKPT} \
        ++model.train_ds.manifest_filepath="$DATA_DIR/hf_manifests/hf_train_manifest.json" \
        ++model.validation_ds.manifest_filepath="$DATA_DIR/hf_manifests/hf_test_manifest.json" \
        ++model.optim.sched.d_model=1024 \
        ++trainer.devices=1 \
        ++trainer.max_epochs=1 \
        ++trainer.limit_train_batches=60 \
        ++trainer.precision=bf16 \
        ++model.train_ds.batch_duration=200 \
        ++model.optim.name="adamw" \
        ++model.optim.lr=0.1 \
        ++model.optim.weight_decay=0.001 \
        ++model.optim.sched.warmup_steps=100 \
        ++exp_manager.version=test \
        ++exp_manager.use_datetime_version=False \
        ++exp_manager.exp_dir=$DATA_DIR/checkpoints

In [ ]:
import nemo.collections.asr as nemo_asr

model = nemo_asr.models.ASRModel.restore_from(
    "/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b.nemo",
    map_location="cpu"
)


In [29]:
prompt_dict = model.cfg.train_ds.prompt_dictionary
print(dict(prompt_dict))

{'en-US': 0, 'en': 0, 'en-GB': 1, 'enGB': 1, 'es-ES': 2, 'esES': 2, 'es-US': 3, 'es': 3, 'zh-CN': 4, 'zh-ZH': 4, 'zh-TW': 5, 'hi-IN': 6, 'hi': 6, 'hi-HI': 6, 'ar-AR': 7, 'ar': 7, 'fr-FR': 8, 'fr': 8, 'de-DE': 9, 'de': 9, 'ja-JP': 10, 'ja-JA': 10, 'ru-RU': 11, 'ru': 11, 'pt-BR': 12, 'pt-PT': 13, 'pt': 13, 'ko-KR': 14, 'ko': 14, 'ko-KO': 14, 'it-IT': 15, 'it': 15, 'nl-NL': 16, 'nl': 16, 'pl-PL': 17, 'pl': 17, 'tr-TR': 18, 'tr': 18, 'uk-UA': 19, 'uk': 19, 'ro-RO': 20, 'ro': 20, 'el-GR': 21, 'el': 21, 'cs-CZ': 22, 'cs': 22, 'hu-HU': 23, 'hu': 23, 'sv-SE': 24, 'sv': 24, 'da-DK': 25, 'da': 25, 'fi-FI': 26, 'fi': 26, 'no-NO': 27, 'no': 27, 'nb-NO': 103, 'nb': 103, 'nn-NO': 104, 'nn': 104, 'sk-SK': 28, 'sk': 28, 'hr-HR': 29, 'hr': 29, 'bg-BG': 30, 'bg': 30, 'lt-LT': 31, 'lt': 31, 'et-EE': 60, 'et': 60, 'lv-LV': 61, 'lv': 61, 'sl-SI': 62, 'sl': 62, 'th-TH': 32, 'vi-VN': 33, 'id-ID': 34, 'ms-MY': 35, 'bn-IN': 36, 'ur-PK': 37, 'fa-IR': 38, 'ta-IN': 39, 'te-IN': 40, 'mr-IN': 41, 'gu-IN': 42, 'kn-I

## Update prompt dict


Audio → FastConformer Encoder → acoustic embedding  (D=1024, T)
                                          +
"mos_Latn" → ID 41 → embedding[41] →  lang vector   (K=128, T)  ← broadcast across time
                                          ↓
                               Concatenate → (D+K=1152, T)
                                          ↓
                               Projection layer → (1024, T)
                                          ↓
                               RNNT Decoder

**Before fine-tuning**

prompt_embedding_table[41] = random noise  ← untrained, useless

**After fine-tuning on mos_Latn data**
prompt_embedding_table[41] = learned vector that represents Mossi acoustics/vocab

**We can clone french embeddings**

embedding_table[41] = embedding_table[7].clone()  # fra_Latn → mos_Latn

- We can make full model load

In [33]:
max_id = max(prompt_dict.values())
print(f"Next available ID: {max_id + 1}")

Next available ID: 105


In [ ]:
from omegaconf import OmegaConf, open_dict

# Add your new language
with open_dict(model.cfg):
    model.cfg.train_ds.prompt_dictionary["mos_Latn"] = max_id + 1
    model.cfg.validation_ds.prompt_dictionary["mos_Latn"] = max_id + 1

# Save patched checkpoint
model.save_to("/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b-mos.nemo")
print("Done!")

- Or just unzip

In [ ]:
import zipfile, yaml, shutil, os

nemo_path = "/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b.nemo"
out_path   = "/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b-mos.nemo"

shutil.copy(nemo_path, out_path)

with zipfile.ZipFile(out_path, "r") as z:
    config_str = z.read("model_config.yaml").decode()

config = yaml.safe_load(config_str)

# Add mos_Latn with next available ID
prompt_dict = config["train_ds"]["prompt_dictionary"]
next_id = max(prompt_dict.values()) + 1
prompt_dict["mos_Latn"] = next_id
config["validation_ds"]["prompt_dictionary"]["mos_Latn"] = next_id

# Write back into the zip
with zipfile.ZipFile(out_path, "a") as z:
    z.writestr("model_config.yaml", yaml.dump(config))

print(f"Added mos_Latn with ID {next_id}")

## Updating the embedding

In [ ]:
import torch
import nemo.collections.asr as nemo_asr
from omegaconf import open_dict

model = nemo_asr.models.ASRModel.restore_from(
    "/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b.nemo",
    map_location="cpu"
)

# ── 1. Check current embedding table ──────────────────────────────
print(model.prompt_encoder)          # find the embedding layer name
print(model.prompt_encoder.weight.shape)  # e.g. (40, 128) → 40 langs, 128-dim

# ── 2. Pick donor language (closest to mos_Latn) ──────────────────
prompt_dict = dict(model.cfg.train_ds.prompt_dictionary)
donor_id  = prompt_dict["fra_Latn"]   # French: same Latin script
new_id    = max(prompt_dict.values()) + 1
print(f"Donor ID: {donor_id}, New ID: {new_id}")

# ── 3. Resize embedding table to add one row ──────────────────────
old_weight = model.prompt_encoder.weight.data        # (N, 128)
donor_vec  = old_weight[donor_id].unsqueeze(0)       # (1, 128)

new_weight = torch.cat([old_weight, donor_vec], dim=0)  # (N+1, 128)

# Replace the embedding layer
num_langs, embed_dim = new_weight.shape
model.prompt_encoder = torch.nn.Embedding(num_langs, embed_dim)
model.prompt_encoder.weight = torch.nn.Parameter(new_weight)

print(f"Embedding table: {old_weight.shape} → {new_weight.shape}")

# ── 4. Register mos_Latn in the config ────────────────────────────
with open_dict(model.cfg):
    for ds in ["train_ds", "validation_ds"]:
        model.cfg[ds].prompt_dictionary["mos_Latn"] = new_id
        model.cfg[ds].prompt_dictionary["mos"]      = new_id  # alias

# ── 5. Save patched checkpoint ────────────────────────────────────
out_path = "/opt/prompt_model/nemotron-3.5-asr-streaming-0.6b-mos.nemo"
model.save_to(out_path)
print(f"Saved to {out_path}")

- Check if it works

In [ ]:
# Reload and confirm
model2 = nemo_asr.models.ASRModel.restore_from(out_path, map_location="cpu")

prompt_dict = dict(model2.cfg.train_ds.prompt_dictionary)
print("mos_Latn ID:", prompt_dict["mos_Latn"])
print("Embedding shape:", model2.prompt_encoder.weight.shape)  # should be (N+1, 128)

# Check the embedding vector was copied from French
mos_vec = model2.prompt_encoder.weight[prompt_dict["mos_Latn"]]
fra_vec = model2.prompt_encoder.weight[prompt_dict["fra_Latn"]]
print("Vectors identical:", torch.allclose(mos_vec, fra_vec))  # True before fine-tuning

- Freeze other language embedding

In [ ]:
def freeze_other_embeddings(grad):
    mask = torch.zeros_like(grad)
    mask[new_id] = 1.0
    return grad * mask

model.prompt_encoder.weight.register_hook(freeze_other_embeddings)

### Verify the existence of the fine-tuning script

### ASR Evaluation

Now that we have a model trained, we need to check how well it performs.

In [ ]:
nemo_file_path = os.path.join(os.environ["DATA_DIR"], "checkpoints/FastConformer-Transducer-BPE-Prompt-Streaming/test/checkpoints/FastConformer-Transducer-BPE-Prompt-Streaming.nemo")
! python /{NEMO_DIR}/examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py \
    model_path={nemo_file_path} \
    dataset_manifest="$DATA_DIR/an4_converted/test_manifest.json" \
    target_lang=auto \
    att_context_size="[56,3]" \
    decoder_type=rnnt \
    pad_and_drop_preencoded=true \
    batch_size=8 \
    cuda=0 \
    strip_lang_tags=false

## More Resources
You can find more information about working with NeMo's ASR models in the [ASR section](https://github.com/NVIDIA/NeMo/tree/main/tutorials/asr) of the NeMo tutorials.

## What's Next?

You can use NeMo to build custom models for your own applications, and deploy them with NVIDIA Riva! Refer to the [Riva NIM deployment tutorial](https://github.com/nvidia-riva/tutorials/blob/main/asr-deploy-am-and-ngram-lm.ipynb).

## Resources

### Process Hugging Face Dataset

If you have an ASR dataset available on Hugging Face (HF), you can load it using the `datasets` library and then convert it into a NeMo manifest file. Below is an example of how to load a dataset from HF and create a NeMo manifest.